In [ ]:
# AgroTrace Folder Inspection and PDF Export

This notebook inspects the AgroTrace repository folder structure, generates a detailed report of all files and subfolders, and exports the report to PDF.

**Note:** If `nbconvert` is not installed or PDF export fails, the notebook will still generate a text report.
</VSCode.Cell>
<VSCode.Cell language="python">
import os
from pathlib import Path
import json
import datetime

base_path = Path(r'c:\Users\chinn\Desktop\AgroTrace')
report_path = base_path / 'agrotrace_folder_report.txt'
pdf_path = base_path / 'agrotrace_folder_report.pdf'

print('Base path:', base_path)
print('Report text path:', report_path)
print('Report PDF path:', pdf_path)


In [ ]:
## Section: Inspect Folder Structure

This section walks the repository directories and gathers metadata for each file and folder.
</VSCode.Cell>
<VSCode.Cell language="python">
from pathlib import Path
import os

items = []
for path in sorted(base_path.rglob('*')):
    try:
        stat = path.stat()
    except OSError:
        continue
    items.append({
        'relative_path': str(path.relative_to(base_path)).replace('\\', '/'),
        'is_dir': path.is_dir(),
        'suffix': path.suffix,
        'size': stat.st_size,
        'modified': datetime.datetime.fromtimestamp(stat.st_mtime).isoformat(),
    })

print('Total items found:', len(items))
items[:10]
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Section: Describe Files and Subdirectories

This section classifies files by type and provides descriptions for key folders and files in the project.
</VSCode.Cell>
<VSCode.Cell language="python">
from collections import Counter

folder_counts = Counter()
file_ext_counts = Counter()
for item in items:
    if item['is_dir']:
        folder_counts[item['relative_path']] += 1
    else:
        file_ext_counts[item['suffix'] or 'NO_EXT'] += 1

print('Top file types:')
for suffix, count in file_ext_counts.most_common(10):
    print(f'  {suffix}: {count}')

print('\nTop folders:')
for folder, count in folder_counts.most_common(10):
    print(f'  {folder}: {count}')
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Section: Generate Detailed Folder Report

Create a structured report describing each top-level folder and its purpose.
</VSCode.Cell>
<VSCode.Cell language="python">
report_lines = []
report_lines.append('AgroTrace Folder Report')
report_lines.append('=' * 40)
report_lines.append(f'Generated: {datetime.datetime.now().isoformat()}')
report_lines.append('')

report_lines.append('Top-level folders and files:')
for child in sorted(base_path.iterdir(), key=lambda p: p.name.lower()):
    if child.is_dir():
        report_lines.append(f'- {child.name}/')
    else:
        report_lines.append(f'- {child.name}')
report_lines.append('')

report_lines.append('Detailed breakdown:')
for child in sorted(base_path.iterdir(), key=lambda p: p.name.lower()):
    if child.is_dir():
        report_lines.append(f'\n{child.name}/')
        report_lines.append('-' * (len(child.name) + 1))
        entries = sorted([p.name for p in child.iterdir()])
        for entry in entries[:50]:
            report_lines.append(f'  - {entry}')
        if len(entries) > 50:
            report_lines.append(f'  - ...and {len(entries) - 50} more entries')
    else:
        report_lines.append(f'\n{child.name}')
        report_lines.append('-' * len(child.name))
        report_lines.append(f'  size: {child.stat().st_size} bytes')
        report_lines.append(f'  modified: {datetime.datetime.fromtimestamp(child.stat().st_mtime).isoformat()}')

with open(report_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(report_lines))

print('Report written to', report_path)
report_lines[:50]
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Section: Save Report as PDF

This section attempts to export the generated report to PDF using `nbconvert`.
</VSCode.Cell>
<VSCode.Cell language="python">
import subprocess

try:
    from nbconvert import PDFExporter
    from traitlets.config import Config
    print('nbconvert is installed, exporting notebook to PDF...')
    c = Config()
    c.PDFExporter.exclude_input = True
    exporter = PDFExporter(config=c)
    exporter.register_preprocessor
    
    nb_path = base_path / 'folder_report.ipynb'
    output, resources = exporter.from_filename(str(nb_path))
    with open(pdf_path, 'wb') as f:
        f.write(output)
    print('PDF written to', pdf_path)
except Exception as e:
    print('PDF export failed:', type(e).__name__, str(e))
    print('A text report is still available at', report_path)
